In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = np.pad(x_train.astype('float32') / 255.0,
                 ((0,0),(2,2),(2,2)), mode='constant')
x_test  = np.pad(x_test.astype('float32') / 255.0,
                 ((0,0),(2,2),(2,2)), mode='constant')

x_train = x_train[..., None]
x_test  = x_test[..., None]

y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

model = keras.Sequential([
    layers.Conv2D(6, (5,5), activation='relu', input_shape=(32,32,1), name='conv1'),
    layers.AveragePooling2D((2,2), strides=2, name='pool1'),
    layers.Conv2D(16, (5,5), activation='relu', name='conv2'),
    layers.AveragePooling2D((2,2), strides=2, name='pool2'),
    layers.Conv2D(120, (5,5), activation='relu', name='conv3'),
    layers.Flatten(),
    layers.Dense(84, activation='relu', name='dense1'),
    layers.Dense(10, activation='softmax', name='output')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test))

def exportar_float(arr, nome):
    flat = arr.flatten()
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"// Dimensoes: {list(arr.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const float {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(f"{v:.8f}f" for v in flat[i:i+8]) for i in range(0, len(flat), 8)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def exportar_int8(arr, nome):
    flat = arr.flatten().astype(np.int8)
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"#include <stdint.h>\n\n"
    cabecalho += f"// Dimensoes: {list(arr.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const int8_t {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(str(v) for v in flat[i:i+16]) for i in range(0, len(flat), 16)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def exportar_uint8(arr, nome):
    flat = arr.flatten().astype(np.uint8)
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"#include <stdint.h>\n\n"
    cabecalho += f"// Dimensoes: {list(arr.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const uint8_t {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(str(v) for v in flat[i:i+16]) for i in range(0, len(flat), 16)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def exportar_escala(escala, nome):
    flat = np.atleast_1d(escala).flatten()
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"// Dimensoes: {list(flat.shape)}\n"
    cabecalho += f"#define {nome}_SIZE {len(flat)}\n"
    cabecalho += f"static const float {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(f"{v:.10f}f" for v in flat[i:i+8]) for i in range(0, len(flat), 8)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def quantizar_pesos(pesos, por_canal=True):
    if por_canal:
        eixo = len(pesos.shape) - 1
        eixos_reduzir = tuple(i for i in range(len(pesos.shape) - 1) if i != eixo)
        max_abs = np.max(np.abs(pesos), axis=eixos_reduzir)
    else:
        max_abs = np.max(np.abs(pesos))
    max_abs = np.maximum(max_abs, 1e-8)

    escala = max_abs / 127.0
    escala_expandida = np.expand_dims(
        escala, axis=tuple(i for i in range(len(pesos.shape) - 1))
    ) if por_canal else escala

    w_int = np.round(pesos / escala_expandida)
    w_int = np.clip(w_int, -127, 127).astype(np.int8)
    return w_int, escala

for camada in ['conv1', 'conv2', 'conv3', 'dense1', 'output']:
    w, b = model.get_layer(camada).get_weights()

    with open(f"{camada}_w_float.h", "w") as f:
        f.write(exportar_float(w, f"{camada}_w_float"))
    with open(f"{camada}_b_float.h", "w") as f:
        f.write(exportar_float(b, f"{camada}_b_float"))

    w_int, escala = quantizar_pesos(w, por_canal=True)
    with open(f"{camada}_w_int8.h", "w") as f:
        f.write(exportar_int8(w_int, f"{camada}_w_int8"))
    with open(f"{camada}_escala.h", "w") as f:
        f.write(exportar_escala(escala, f"{camada}_escala"))

img_teste = (x_test[0] * 255.0).astype(np.uint8)
with open("imagem_teste_uint8.h", "w") as f:
    f.write(exportar_uint8(img_teste, "imagem_teste_uint8"))

print("Exportação concluída: float, int8 (com escala por canal) e imagem gerados do mesmo modelo.")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (AveragePooling2D)        │ (None, 14, 14, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (AveragePooling2D)        │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 33s 65ms/step - accuracy: 0.8880 - loss: 0.3777 - val_accuracy: 0.9642 - val_loss: 0.1182
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 67ms/step - accuracy: 0.9696 - loss: 0.0994 - val_accuracy: 0.9786 - val_loss: 0.0646
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 69ms/step - accuracy: 0.9801 - loss: 0.0649 - val_accuracy: 0.9842 - val_loss: 0.0470
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 30s 63ms/step - accuracy: 0.9848 - loss: 0.0501 - val_accuracy: 0.9868 - val_loss: 0.0403
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.9875 - loss: 0.0414 - val_accuracy: 0.9870 - val_loss: 0.0373
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 42s 66ms/step - accuracy: 0.9898 - loss: 0.0349 - val_accuracy: 0.9886 - val_loss: 0.0331
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 67ms/step - accuracy: 0.9914 - loss: 0.0296 - val_accuracy: 0.9902 - val_loss: 0.0298
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 66ms/step - accuracy: 0.9922 - loss: 0.0260 - 

In [ ]:
import numpy as np

imgs_teste = (x_test * 255.0).astype(np.uint8)
rotulos_teste = np.argmax(y_test, axis=1).astype(np.uint8)

# exporta imagens
def exportar_uint8_dataset(imgs, nome):
    flat = imgs.flatten()
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"#include <stdint.h>\n\n"
    cabecalho += f"// Dimensoes: {list(imgs.shape)}\n"
    cabecalho += f"#define {nome}_NUM_IMAGENS {imgs.shape[0]}\n"
    cabecalho += f"#define {nome}_TAMANHO {len(flat)}\n"
    cabecalho += f"static const uint8_t {nome}[{len(flat)}] = {{\n    "
    linhas = [", ".join(str(v) for v in flat[i:i+16]) for i in range(0, len(flat), 16)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

def exportar_rotulos(rotulos, nome):
    cabecalho = f"#ifndef {nome}_H\n#define {nome}_H\n\n"
    cabecalho += f"#include <stdint.h>\n\n"
    cabecalho += f"#define {nome}_TAMANHO {len(rotulos)}\n"
    cabecalho += f"static const uint8_t {nome}[{len(rotulos)}] = {{\n    "
    linhas = [", ".join(str(v) for v in rotulos[i:i+20]) for i in range(0, len(rotulos), 20)]
    cabecalho += ",\n    ".join(linhas)
    cabecalho += "\n};\n\n#endif\n"
    return cabecalho

with open("x_test_uint8.h", "w") as f:
    f.write(exportar_uint8_dataset(imgs_teste, "x_test_uint8"))

with open("y_test_uint8.h", "w") as f:
    f.write(exportar_rotulos(rotulos_teste, "y_test_uint8"))

print("Dataset de teste exportado com sucesso")

Dataset de teste exportado com sucesso


In [ ]:
# @title
import subprocess
subprocess.run(
    "zip -q pesos_lenet5_int8.zip *_w_int8.h *_escala.h *_b_float.h imagem_teste_uint8.h",
    shell=True, check=True
)
print("pesos_lenet5_int8.zip criado.")

try:
    from google.colab import files
    files.download('pesos_lenet5_int8.zip')
except ImportError:
    pass

pesos_lenet5_int8.zip criado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pred = model.predict(x_test[0:1])
print(pred[0])
print("Classe prevista:", np.argmax(pred))
print("Confiança: {:.2f}%".format(100 * np.max(pred)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
[2.7961680e-09 1.9830109e-07 1.0271559e-07 4.2998383e-07 7.9094787e-13
 2.7668102e-08 4.1522097e-14 9.9999905e-01 3.1697322e-08 9.0970772e-08]
Classe prevista: 7
Confiança: 100.00%
